# Pareto front from the multi-task GP

Refits a multi-task GP to every point the search measured, then reads the Pareto-optimal
set off the posterior mean.

The kernel is Matern-1.5, matching `sparsest.search`, so the posterior plotted here is the
same surrogate the search used to choose query points. Targets are standardised before
fitting and mapped back afterwards, which keeps the two objectives on comparable scales.

In [ ]:
# Resolve every path through the package config, so this notebook follows whatever
# SPARSEST_DATA_ROOT / SPARSEST_OUTPUT_ROOT point at. Switch datasets here.
CONFIG_NAME = "ipad"   # or "mnist"

import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from sparsest.config import load_config
cfg = load_config(REPO / "configs" / f"{CONFIG_NAME}.yaml")
print("dataset:", cfg.dataset, "| window:", cfg.window)
print("outputs:", cfg.output_path("checkpoints", create=False).parent)

In [ ]:
import torch
import gpytorch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from botorch.utils.multi_objective.pareto import is_non_dominated

FIG_DIR = cfg.output_path("figures")

In [ ]:
class MultitaskGPModel(gpytorch.models.ExactGP):
    """Two correlated outputs (MSE, occupancy) over the preference weight."""

    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.ConstantMean(), num_tasks=2
        )
        self.covar_module = gpytorch.kernels.MultitaskKernel(
            gpytorch.kernels.MaternKernel(nu=1.5), num_tasks=2, rank=1
        )

    def forward(self, x):
        return gpytorch.distributions.MultitaskMultivariateNormal(
            self.mean_module(x), self.covar_module(x)
        )

In [ ]:
# Load the most recent GP checkpoint written by the search; it carries the full
# measurement set in X_train / Y_train.
gp_dir = cfg.output_path("gp_models", create=False)
gp_files = sorted(gp_dir.glob("gp_iter_*.pth"),
                  key=lambda p: int(p.stem.split("_")[-1]))
assert gp_files, f"no GP checkpoints in {gp_dir}; run the search first"

checkpoint = torch.load(gp_files[-1], map_location="cpu")
train_x, train_y = checkpoint["X_train"], checkpoint["Y_train"]
print(f"{gp_files[-1].name}: {train_x.shape[0]} measured weights")

# Drop points here if a run is known bad, e.g. DROP = [0].
DROP = []
if DROP:
    keep = torch.tensor([i for i in range(train_x.size(0)) if i not in DROP])
    train_x, train_y = train_x[keep], train_y[keep]

for w, (mse, occ) in zip(train_x.flatten().tolist(), train_y.tolist()):
    print(f"  w={w:.3f}  mse={mse:.5f}  occupancy={occ:.5f}")

In [ ]:
Y_mean, Y_std = train_y.mean(dim=0), train_y.std(dim=0)
train_y_norm = (train_y - Y_mean) / Y_std

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=2)
model = MultitaskGPModel(train_x, train_y_norm, likelihood)
model.train(); likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

log = defaultdict(list)
for i in range(200):
    optimizer.zero_grad()
    loss = -mll(model(train_x), train_y_norm)
    loss.backward()
    optimizer.step()
    log["iter"].append(i); log["-mll"].append(loss.item())

print(f"final -MLL: {log['-mll'][-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(log["iter"], log["-mll"])
ax.set_xlabel("Iteration"); ax.set_ylabel("Negative MLL")
ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIG_DIR / f"gp_training_{CONFIG_NAME}.pdf")

In [ ]:
model.eval(); likelihood.eval()
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    w_grid = torch.linspace(0, 1, 200)
    pred = likelihood(model(w_grid))
    mean = pred.mean
    lower, upper = pred.confidence_region()

# is_non_dominated maximises, and we want both objectives small, so negate.
mask = is_non_dominated(-mean)
pareto = mean[mask] * Y_std + Y_mean
w_pareto = w_grid[mask]
lower_p, upper_p = lower[mask] * Y_std + Y_mean, upper[mask] * Y_std + Y_mean

# Full posterior on the original scale, for the fit-quality panels.
mean_full = mean * Y_std + Y_mean
lower_full, upper_full = lower * Y_std + Y_mean, upper * Y_std + Y_mean

print(f"{mask.sum().item()} of {len(w_grid)} grid points are Pareto-optimal")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, col, title in [(ax1, 0, "MSE"), (ax2, 1, "Occupancy")]:
    ax.plot(train_x.numpy(), train_y[:, col].numpy(), "k*", label="Observed")
    ax.plot(w_grid.numpy(), mean_full[:, col].numpy(), "b", label="Mean")
    ax.fill_between(w_grid.numpy(), lower_full[:, col].numpy(),
                    upper_full[:, col].numpy(), alpha=0.4, label="Confidence")
    ax.set_xlabel(r"Preference weight $w_{\mathrm{MSE}}$", fontsize=13)
    ax.set_title(f"Predictions on {title}", fontsize=13)
    ax.legend(fontsize=10)

fig.tight_layout()
fig.savefig(FIG_DIR / f"fitting_quality_{CONFIG_NAME}.pdf")

In [ ]:
# Optional: clip the tail where MSE explodes, so the interesting region is legible.
MSE_LIMIT = None   # e.g. 0.008

p, tx, ty = pareto, train_x, train_y
if MSE_LIMIT is not None:
    p = p[p[:, 0] < MSE_LIMIT]
    keep = ty[:, 0] < MSE_LIMIT
    tx, ty = tx[keep], ty[keep]

fig, ax = plt.subplots(figsize=(7, 5))
order = torch.argsort(p[:, 0])
ax.plot(p[order, 0].numpy(), p[order, 1].numpy(),
        label="Approximated Pareto front", color="blue", linewidth=2)
sc = ax.scatter(ty[:, 0], ty[:, 1], c=tx, cmap="inferno_r",
                s=50, edgecolor="black", zorder=5, label="Evaluated runs")

fig.colorbar(sc, ax=ax).set_label(r"Preference weight $w_{\mathrm{MSE}}$", fontsize=11)
ax.set_xlabel("MSE", fontsize=12); ax.set_ylabel("Occupancy", fontsize=12)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.3)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / f"pareto_front_{CONFIG_NAME}.pdf")
print("wrote", FIG_DIR / f"pareto_front_{CONFIG_NAME}.pdf")